# REGRESIÓN LOGÍSTICA
### **Nombre:** Carlos Enrique Nieves Ochoa
### **Expediente:** 758210

Utilizamos el dataset de Heart Prediction.

In [31]:
# Carga de librerías
import pandas as pd
from sklearn.linear_model import LogisticRegression

In [32]:
# Carga del archivo
df = pd.read_csv("Heart Prediction Quantum Dataset (1).csv")
df.head()

,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
0,68,Hombre,105.00,191,107,1
1,58,Mujer,97.00,249,89,0
2,44,Mujer,93.00,190,82,1
3,72,Hombre,93.00,183,101,1
4,37,Mujer,145.00,166,103,1


## Regresión logística con la presión arterial

Se utiliza BloodPressure como factor y HeartDisease como respuesta.

In [33]:
# Cambiar la coma decimal por punto y convertir la presión a número
df["BloodPressure"] = df["BloodPressure"].str.replace(",", ".", regex=False)
df["BloodPressure"] = df["BloodPressure"].astype(float)

In [34]:
# Separar el factor y la variable de respuesta
X = df[["BloodPressure"]]
y = df["HeartDisease"]

In [35]:
# Ajustar la regresión logística
modelo = LogisticRegression()
modelo.fit(X, y)

b0 = modelo.intercept_[0]
b1 = modelo.coef_[0][0]

print("Intercepto:", b0)
print("Coeficiente de BloodPressure:", b1)
print(f"logit(p) = {b0:.6f} {b1:+.6f}(BloodPressure)")

Intercepto: 1.6217289935252623
Coeficiente de BloodPressure: -0.00911071266667734
logit(p) = 1.621729 -0.009111(BloodPressure)


In [36]:
# Probabilidades del modelo original
prob = modelo.predict_proba(X)
pd.DataFrame(prob, columns=["Probabilidad de 0", "Probabilidad de 1"])

,Probabilidad de 0,Probabilidad de 1
0,0.339594,0.660406
1,0.323444,0.676556
2,0.315521,0.684479
3,0.315521,0.684479
4,0.425392,0.574608
...,...,...
495,0.383722,0.616278
496,0.468150,0.531850
497,0.456826,0.543174
498,0.343692,0.656308


**Observación** -> logit(p) es lo mismo que escribir ln(P/1 - P)

### Modificando el parámetro de C (inverso de la fuerza de la regularización)

## Modelo con C = 0.000001

In [37]:
modelo = LogisticRegression(C=0.000001)
modelo.fit(X, y)

b0 = modelo.intercept_[0]
b1 = modelo.coef_[0][0]

print("Intercepto:", b0)
print("Coeficiente de BloodPressure:", b1)
print(f"logit(p) = {b0:.6f} {b1:+.6f}(BloodPressure)")

Intercepto: 0.4982272041003548
Coeficiente de BloodPressure: -0.0006978645382854834
logit(p) = 0.498227 -0.000698(BloodPressure)


## Modelo con C = 1000000

In [38]:
modelo = LogisticRegression(C=1000000)
modelo.fit(X, y)

b0 = modelo.intercept_[0]
b1 = modelo.coef_[0][0]

print("Intercepto:", b0)
print("Coeficiente de BloodPressure:", b1)
print(f"logit(p) = {b0:.6f} {b1:+.6f}(BloodPressure)")

Intercepto: 1.6217439478286597
Coeficiente de BloodPressure: -0.009110824174604
logit(p) = 1.621744 -0.009111(BloodPressure)


## Cálculo de las métricas threshold (0.3, 0.5 y 0.8)

In [39]:
# Usar todas las variables como factores
X2 = df.drop(columns=["HeartDisease"])
y2 = df["HeartDisease"]

# Unificar Gender y codificar hombre = 0, mujer = 1
X2["Gender"] = X2["Gender"].str.strip().str.lower()
X2["Gender"] = X2["Gender"].map({
    "hombre": 0,
    "hombre}": 0,
    "ombre": 0,
    "mujer": 1,
    "mujer{": 1
})

# Ajustar otro modelo con C = 1 por defecto
modelo2 = LogisticRegression()
modelo2.fit(X2, y2)

# Obtener la probabilidad de la clase 1
prob2 = modelo2.predict_proba(X2)[:, 1]

# Calcular las métricas para cada umbral
res2 = []

for t in [0.3, 0.5, 0.8]:
    y_pred2 = (prob2 >= t).astype(int)

    tp = ((y2 == 1) & (y_pred2 == 1)).sum()
    tn = ((y2 == 0) & (y_pred2 == 0)).sum()
    fp = ((y2 == 0) & (y_pred2 == 1)).sum()
    fn = ((y2 == 1) & (y_pred2 == 0)).sum()

    acc = (tp + tn) / (tp + tn + fp + fn)
    rec = tp / (tp + fn)
    pre = tp / (tp + fp) if tp + fp > 0 else float("nan")
    esp = tn / (tn + fp)

    res2.append([t, tp, tn, fp, fn, acc, rec, pre, esp])

tabla2 = pd.DataFrame(
    res2,
    columns=[
        "Umbral", "TP", "TN", "FP", "FN",
        "Accuracy", "Recall", "Precisión", "Especificidad"
    ]
)

tabla2

,Umbral,TP,TN,FP,FN,Accuracy,Recall,Precisión,Especificidad
0,0.3,281,73,127,19,0.708,0.936667,0.688725,0.365
1,0.5,238,127,73,62,0.730,0.793333,0.765273,0.635
2,0.8,134,184,16,166,0.636,0.446667,0.893333,0.920


Con el umbral de 0.3, el modelo necesita una probabilidad relativamente baja para predecir la clase 1. Detecta 281 de los 300 casos positivos, pero también clasifica como positivos 127 casos que realmente eran negativos. Por eso tiene un recall alto, de 93.67%, aunque genera muchos falsos positivos.

Con el umbral de 0.5, obtiene un 73% de aciertos, el mayor accuracy de los tres umbrales. Con el umbral de 0.8, exige una probabilidad más alta para predecir la clase 1 y los falsos positivos bajan a 16. Su precisión llega a 89.33%, lo que significa que ese porcentaje de los casos predichos como positivos realmente lo era. Sin embargo, deja sin detectar 166 casos positivos.

Bajar el umbral permite detectar más positivos, mientras que subirlo reduce las falsas alarmas, pero deja pasar más positivos reales. Elegir el umbral depende de cuál de esos errores se quiera reducir.